# 评价前OOF均值配置对照
只用2020–2022的756日OOF配置均值，不优化比例。保留完整风险、恢复与每日分类输出，采用同样每天执行目标规则；3个成本独立账户，对比逐日动态配置。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.workflow.minute_pit_allocator import _ensemble
from src.workflow.pit_constituent_auxiliary_training_allocation import _smooth_weights
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.backtest.full_path_tiered import replay_full_path
from src.backtest.stage1_ibkr import native_tiered_stage1
from src.backtest.qlib_continuous_position import QlibContinuousPosition
from src.analysis.integrated_allocation import paired_metrics,annual_paired_metrics,stationary_bootstrap_excess
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/oof_mean_allocation_20261015';plan=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013'
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(provider),region='us')
frozen,fa=_source(root,plan['frozen_source'],['inputs/plan','analysis/predictions','analysis/targets','models/candidate_71',*[f'analysis/native/soxx_{v}' for v in plan['slippage_bps']]])
oof,oa=_source(root,plan['oof_source'],['analysis/oof_predictions']);baseline,ba=_source(root,plan['always_source'],['inputs/plan',*[f'analysis/native/strong_always_{v}' for v in plan['slippage_bps']]])
source,sa=_source(root,plan['source'],['inputs/frame']);controller,ca=_source(root,plan['price_source'],['inputs/controller']);action,aa=_source(root,plan['action_source'],['inputs/plan','inputs/market','controller/source_recovery_selection'])
for existing in [frozen,baseline]:
 for key in ['tariff','initial_cash','start','end','as_of','slippage_bps','maximum_soxl_weight','minimum_cash_weight','maximum_drawdown']:assert existing['inputs/plan'][key]==plan[key]
frame=source['inputs/frame'];frame=frame[frame.signal_date.between(plan['start'],plan['end'])].copy();dates=pd.DatetimeIndex(frame.signal_date);market=action['inputs/market'];market_dates=pd.DatetimeIndex(market.signal_date);model=frozen['models/candidate_71'];raw=_ensemble(frozen['analysis/predictions'],'candidate',plan['seeds']).reindex(market_dates);returns=market.set_index('signal_date')[['soxl_return','soxx_return']]
reconstructed=_smooth_weights(raw,spec=model.spec,candidate=model.candidate,asset_returns=returns);np.testing.assert_allclose(reconstructed,frozen['analysis/targets']['candidate'],atol=1e-10,rtol=1e-10)
weights=['soxl_weight','soxx_weight','cash_weight'];past=oof['analysis/oof_predictions'];assert len(past)==756 and past.trade_date.max()<pd.Timestamp('2023-01-01');assert np.isfinite(past[weights]).all().all();mean=past[weights].mean()
assert np.isclose(mean.sum(),1,atol=1e-12) and mean.soxl_weight<=plan['maximum_soxl_weight'] and mean.cash_weight>=plan['minimum_cash_weight']
constant=raw.copy()
for name in weights:constant[name]=mean[name]
pd.testing.assert_frame_equal(raw.drop(columns=weights),constant.drop(columns=weights));targets=_smooth_weights(constant,spec=model.spec,candidate=model.candidate,asset_returns=returns).reindex(dates);assert targets.notna().all().all()
for name in targets:frame[name]=targets[name].to_numpy()
recovery=IntradayRecoveryDirectAllocationSpec(root=root,**action['inputs/plan']['recovery_spec']);spec=Stage1OpenActionValueSpec(root=root,**action['inputs/plan']['action_spec']);modified=dict(controller['inputs/controller']);modified['frozen_open_action_value_model']=None
with R.start(experiment_name=card.name,uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=plan,source_audits={'frozen':fa,'oof':oa,'baseline':ba,'market':sa,'controller':ca,'action':aa},mean_weights=mean,frame=frame,raw_predictions=constant);rec.log_params(new_fits=0,maximum_new_native_cases=3)
books={}
for cost in plan['slippage_bps']:
 replay=replay_full_path(frame,models=None,plan=plan,slippage_bps=cost,controller=modified,recovery_spec=recovery,action_spec=spec,selection=action['controller/source_recovery_selection']);assert replay['replay'].execute_open_target.all()
 book=native_tiered_stage1(frame,replay,provider_uri=provider,tariff=plan['tariff'],slippage_bps=cost,initial_cash=plan['initial_cash'],freq='1min',opening_offset=pd.Timedelta(hours=9,minutes=47),position_type=QlibContinuousPosition);books[('fixed_oof_mean',cost)]=book;books[('dynamic',cost)]=baseline[f'analysis/native/strong_always_{cost}'];books[('soxx',cost)]=frozen[f'analysis/native/soxx_{cost}'];rec.save_objects(artifact_path='analysis/native',**{f'{family}_{cost}':books[(family,cost)] for family in ['fixed_oof_mean','dynamic','soxx']})
rows=[];annual=[];uncertainty=[]
for (family,cost),book in books.items():
 path=book['authority'];bench=books[('soxx',cost)]['authority'];relative=np.log1p(path.strategy_return)-np.log1p(bench.strategy_return)
 rows.append({'family':family,'slippage_bps':cost,**paired_metrics(path,bench),'monitored_drawdown_pct':-100*path.monitored_drawdown.max(),'mean_cash_pct':100*path.ending_cash_weight.mean(),'mean_nominal_beta':(3*path.ending_soxl_weight+path.ending_soxx_weight).mean(),'orders':len(book['orders']),'positive_126d_fraction':relative.rolling(126).sum().dropna().gt(0).mean()});annual.append(annual_paired_metrics(path,bench).assign(family=family,slippage_bps=cost))
 if family=='fixed_oof_mean':
  for comparator in ['soxx','dynamic']:uncertainty.append({'slippage_bps':cost,'comparator':comparator,**stationary_bootstrap_excess(path,books[(comparator,cost)]['authority'],mean_block=63,resamples=2000,alpha=.025,seed=20261015)})
table=pd.DataFrame(rows);annual=pd.concat(annual,ignore_index=True);uncertainty=pd.DataFrame(uncertainty);cand=table[table.family.eq('fixed_oof_mean')].set_index('slippage_bps');dynamic=table[table.family.eq('dynamic')].set_index('slippage_bps')
gates={'net_soxx_excess_all_costs':bool(cand.relative_wealth.gt(1).all()),'beat_dynamic_all_costs':bool(cand.terminal_wealth.gt(dynamic.terminal_wealth).all()),'drawdown_30pct':bool(cand.max_drawdown_pct.ge(-30).all() and cand.monitored_drawdown_pct.ge(-30).all()),'rolling_70pct':bool(cand.positive_126d_fraction.ge(.7).all()),'annual_70pct':bool(annual[annual.family.eq('fixed_oof_mean')].groupby('slippage_bps').relative_wealth.apply(lambda v:v.gt(1).mean()).ge(.7).all()),'bootstrap_positive':bool(uncertainty.lower_bound_positive.all()),'independent_validation':False}
policy={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/native/*']};summary={'experiment':card.name,'decision':'not_promoted_fixed_oof_mean','gates':gates,'new_fits':0,'new_native_cases':3,'mean_weights':mean.to_dict(),'provider_uri':str(provider),'region':'us','as_of':plan['as_of'],'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy,'objective_achieved':False,'scope':plan['scope']}
rec.save_objects(artifact_path='analysis',comparison=table,annual=annual,uncertainty=uncertainty,summary=summary);rec.log_metrics(new_native_cases=3,registered_artifact_count=18)
for name,value in [('backtest_benchmark_comparison',table),('annual_metrics',annual),('uncertainty',uncertainty)]:value.to_csv(card/(name+'.csv'),index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'artifact_policy':policy}))
binding=json.loads((root/'research/open_router_ablation_20261015/nas_binding.json').read_text());binding.pop('evidence',None);(card/'nas_binding.json').write_text(json.dumps(binding,indent=2))
a=Path(summary['recorder_local_dir'])/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(f.relative_to(card)),'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in sorted(a.rglob('*')) if f.is_file()]},indent=2))
summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2));print(mean);print(table[['family','slippage_bps','return_pct','monitored_drawdown_pct']].to_string(index=False))


# 评价前固定均值配置：不能替代逐日配置

完成0拟合、3个新增完整Qlib原生账户。固定权重来自2020–2022全部756日年度OOF预测的等日算术均值：SOXL27.471629%、SOXX46.688953%、现金25.839418%。最大来源交易日为2022-12-30，没有用2023+评价预测或收益计算权重，没有优化比例、挑年份或搜索窗口。

只替换原始配置三权重，每日tail/alpha分类输出、原EMA与目标保持逻辑、风险/恢复规则仍保持；与较强历史参考一样每天执行风险投影后的目标。三个成本各自连续持仓、现金、净值及风险状态。原逐日目标经过同样平滑重构，与原Recorder目标逐值核对通过；固定配置的非权重输出也核对未改。该对照仍含分类输出及控制规则，不是纯被动组合。

10万美元、同一IBKR Pro Tiered费用投影，2023-01-03信号至2026-10-09退出，共944区间。

| 方案 | 0bp净收益 | 2bp净收益 | 5bp净收益 | 2bp监控最大回撤 |
|---|---:|---:|---:|---:|
| 评价前OOF均值配置 | +112.27% | +107.18% | +135.19% | 31.82% |
| 原逐日配置、每天执行 | +358.11% | +359.33% | +351.07% | 27.71% |
| SOXX | +387.75% | +387.73% | +387.70% | 43.39% |

固定配置三个成本均低于逐日配置和SOXX，监控回撤31.40%/31.82%/33.35%均超过30%。2bp实际平均名义beta为0.7506，逐日配置为0.9148；固定原始比例并不固定实际敞口，风险状态会改变持仓。不得把两账户巨大收益差全归为动态模型的预测alpha，也不能将OOF均值叫作风险匹配对照。名义beta为3×SOXL权重+SOXX权重，不是回归估计。

与旧tangent_mean_allocation_20261014不同，本轮均值在评价开始前可计算，且保留完整风险层；旧卡使用评价期事后均值并旁路动态风险，不可把两卡差异归为单一因素。研究结构依旧经过复用历史选择，这不是独立验证。

决策not_promoted_fixed_oof_mean。固定均值路线关闭，不改加权方法、不换年份、不调比例或风险阈值。逐日配置仍是较好的历史研究参考，但并未取得稳定净SOXX超额。后续需要分清预测信息与内生风险状态的贡献，优先用训练期时间外预测与未来收益的条件关系确定信息缺口；不能仅因胜过此低收益对照就继续扩大模型或宣称择时有效。

所有新账户通过原生Qlib费用/持仓/净值对账，NAS冷恢复逐对象哈希通过。年度、滚动及区块置信结果随卡保存，生产未改，目标未完成。
